In [ ]:

import itertools
import os

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_recall_fscore_support,
)

# Paths and settings
DIM = 32
BASE = "/"
IN_PT = ""

RATIOS = (0.7, 0.2, 0.1)
TRAIN_NORMAL_ONLY = True
SEED = 42

GRID = {
    "n_estimators": [300],
    "max_samples": [256, 1024],
    "max_features": [0.3, 0.6, 1.0],
}

# Load and validate the gathered features
if not os.path.isfile(IN_PT):
    raise FileNotFoundError(f"Input file not found: {IN_PT}")
if abs(sum(RATIOS) - 1.0) > 1e-9:
    raise ValueError(f"Split ratios must sum to 1: {RATIOS}")

d = torch.load(IN_PT, map_location="cpu", weights_only=False)
X = d["X"].numpy().astype(np.float64)
MASK = d["mask"].numpy()
y = d["label"].numpy().astype(int)
names = list(d["feature_names"])
offsets = d["user_offsets"]
NSTAT, MODS = int(d["n_stat"]), list(d["modalities"])
N, D = X.shape

if int(d["n_components"]) != DIM:
    raise ValueError("Embedding dimension mismatch")
if len(names) != D or MASK.shape != X.shape:
    raise ValueError("Feature names or mask shape mismatch")
if X[~MASK].any() or not np.isfinite(X).all():
    raise ValueError("Invalid feature values")

# Feature sets
STAT_IDX = list(range(NSTAT))
FLAG_IDX = [names.index(f"{m}_missing") for m in MODS]
EMB_IDX = [i for i in range(D) if i not in set(STAT_IDX + FLAG_IDX)]
FEATURE_SETS = {
    "stat": STAT_IDX,
    "stat+flag": STAT_IDX + FLAG_IDX,
    "emb+flag": EMB_IDX + FLAG_IDX,
    "all": list(range(D)),
}

# Chronological train/validation/test split for each user
tr, va, te = [], [], []
for u in d["user_order"]:
    s, e = offsets[u]
    n = e - s
    c1 = s + int(round(n * RATIOS[0]))
    c2 = s + int(round(n * (RATIOS[0] + RATIOS[1])))
    tr += list(range(s, c1))
    va += list(range(c1, c2))
    te += list(range(c2, e))

tr, va, te = map(np.array, (tr, va, te))
tr_fit = tr[y[tr] == 0] if TRAIN_NORMAL_ONLY else tr

if len(set(tr) & set(va)) or len(set(va) & set(te)) or len(set(tr) & set(te)):
    raise ValueError("Overlapping split indices")
if len(tr) + len(va) + len(te) != N:
    raise ValueError("Split sizes do not match the dataset")
if y[va].sum() == 0 or y[te].sum() == 0:
    raise ValueError("Validation and test sets must contain positive labels")


def pick_threshold(y_true, score):
    """Select the threshold maximizing validation F1."""
    prec, rec, thr = precision_recall_curve(y_true, score)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return float(thr[int(np.nanargmax(f1[:-1]))])


def evaluate(y_true, score, thr):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, (score >= thr).astype(int), average="binary", zero_division=0
    )
    return {
        "precision": p,
        "recall": r,
        "F1": f,
        "PR_AUC": average_precision_score(y_true, score),
    }


# Hyperparameter search based on validation PR-AUC
combos = [dict(zip(GRID, v)) for v in itertools.product(*GRID.values())]
search, cache = [], {}

for tag, cols in FEATURE_SETS.items():
    for pr in combos:
        clf = IsolationForest(
            **pr, contamination="auto", random_state=SEED, n_jobs=-1
        ).fit(X[np.ix_(tr_fit, cols)])
        s = -clf.score_samples(X[:, cols])
        search.append({
            "feature_set": tag,
            "n_features": len(cols),
            **pr,
            "val_PR_AUC": average_precision_score(y[va], s[va]),
        })
        cache[(tag, tuple(pr.items()))] = s

best = max(search, key=lambda r: r["val_PR_AUC"])
print(f"Selected: {best['feature_set']}, val PR-AUC={best['val_PR_AUC']:.4f}")

# Test evaluation with thresholds selected on validation F1
rows = []


def add(name, score, nfeat):
    thr = pick_threshold(y[va], score[va])
    rows.append({
        "model": name,
        "n_features": nfeat,
        **evaluate(y[te], score[te], thr),
    })


for tag in FEATURE_SETS:
    b = max(
        (r for r in search if r["feature_set"] == tag),
        key=lambda r: r["val_PR_AUC"],
    )
    mark = " *" if tag == best["feature_set"] else ""
    add(f"IF[{tag}]{mark}", cache[(tag, tuple((k, b[k]) for k in GRID))], b["n_features"])

add("trivial[file활동]", 1.0 - X[:, names.index("file_missing")], 1)

res = (
    pd.DataFrame(rows)
    .set_index("model")
    [["n_features", "precision", "recall", "F1", "PR_AUC"]]
    .sort_values("PR_AUC", ascending=False)
    .round(4)
)

print("\nTEST RESULTS")
print(res.to_string())
print("* = best configuration selected by validation PR-AUC")
